# 251114 - Learning about Spark

## Pandas APIs on Spark
### Scaling pandas with Spark
Pandas DataFrames vs Spark DataFrames:
- Pandas DataFrames are mutable, eagerily evaluated and maintain row order. They are restricted to a single machine. They perform well with small datasets
- Spark DataFrames are distributed, lazily evaluated, inmutable, do not maintain row order. They perform well with big datasets
- Pandas APIs on Spark (aka Pyspark.pandas) is the way to use pandas syntax on Spark DataFrames. However, it is not as efficient as implemting the solution natively in Spark

In [0]:
import pandas as pd
# Remember how to read in CSV from pandas
file_path = "/Workspace/Users/almayo@gmail.com/datasets/ticket_details.csv"
df_pandas = pd.read_csv(file_path)
display(df_pandas.head())

# Pandas APIs on Spark
import pyspark.pandas as ps
# Task: Read CSV file
"""
OBS! A pre-requisite is to upload the CSV file to a Unity Catalog volume and use the volume path
First I had to create a volume called 'learning' under Catalog/workspace/default and then upload the CSV file to the volume
"""
file_path = "/Volumes/workspace/default/learning/ticket_details.csv"
df_pandas_on_spark = ps.read_csv(file_path, inferSchema=True, multiLine=True, escape='"')
display(df_pandas_on_spark.head())


#df_weather = spark.table('samples.accuweather.forecast_daily_calendar_imperial')
#display(df_weather)

## Converting from "pandas API on Spark" df to Spark df

In [0]:
df_spark = df_pandas_on_spark.to_spark()
display(df_spark)

## Converting from Spark df to "pandas API on Spark" df
This conversion might be useful to use some of the pandas methods and get, for example, visualizations or aggregations on Series

In [0]:
# Alt. 1:
df_pandas_on_spark2 = ps.DataFrame(df_spark)
# Alt. 2:
# df_pandas_on_spark2 = df_spark.pandas_api()
display(df_pandas_on_spark2.head())

## SQL on 'pandas API on Spark' df

In [0]:
import pyspark.pandas as ps
import pandas as pd

file_path = "/Workspace/Users/almayo@gmail.com/datasets/ticket_details.csv"
df_pandas = pd.read_csv(file_path)

df_pandas_on_spark = ps.from_pandas(df_pandas)
df_pandas_on_spark.to_table("df_pandas_view")

result = ps.sql("SELECT * FROM df_pandas_view")
display(result.head())

# 251115 PySpark Basics
https://docs.databricks.com/aws/en/pyspark/basics

## Import SQL functions and data types

In [0]:
# import select functions and types
from pyspark.sql.types import IntegerType, StringType
from pyspark.sql.functions import floor, round

# import modules using an alias
import pyspark.sql.types as T
import pyspark.sql.functions as F

## Data Definition Language (DDL): Create a DataFrame

### Create a DataFrame with specified values

In [0]:
"""
Method: spark.createDataFrame()
Args:
  - data: defined as a list of nested tuples
  - schema:
    - Alt. 1: 'Simple schema': list of column names. The data types are automatically infered.
    - Alt. 2: 'StructType': list of StructField objects
"""
# Alt.1
df_children = spark.createDataFrame(
  data = [("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
  schema = ['name', 'age'])
display(df_children)

# Alt.2
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

df_children_with_schema = spark.createDataFrame(
  data = [("Mikhail", 15), ("Zaky", 13), ("Zoya", 8)],
  schema = StructType([
    StructField('name', StringType(), True), # True = Nullable
    StructField('age', IntegerType(), True)
  ])
)
display(df_children_with_schema)

### Create a DataFrame from a table in Unity Catalog
For learning purposes, we may find several sample tables under Catalog/Delta Shares Received/samples/accuweather/<multiple_tables>

In [0]:
# Find the table path by righ click on the table to access the table details
df_weather = spark.table('samples.accuweather.forecast_daily_calendar_imperial')
display(df_weather) 

### Create a DataFrame from an uploaded file
As a pre-req, the file must be first uploaded to a Unity Catalog volume.
Steps:
1.- Create a volume from the Catalog Explorer menu under the schema (db icon) that you want
2.- Navigate to the new or existing volume
3.- Upload the file (e.g. CSV file) to the volume ("Upload to this volume" button)
4.- Copy volume path
5.- Call the spark.read() method

In [0]:
volume_file_path = "/Volumes/workspace/default/learning/ticket_details.csv"

df_csv = (spark.read
  .format("csv")
  .option("header", True)
  .option("inferSchema", True)
  .load(volume_file_path)
)
# OBS! En estricto orden secuencial, primero spark.read() lee el CSV file y luego se crea un Spark DataFrame
display(df_csv)

### Bonus: Create a Delta Table from a Spark DataFrame

In [0]:
"""
(df_csv
 .write
 .mode("overwrite")
 .format("delta")
 .save("/Volumes/workspace/default/learning/ticket_details.delta")
 )
 #OBS! Esto ha creado un Delta Table as a file en el volume. Es un directorio que incluye un directorio _delta_log
"""
 # Create a Delta Table in the metastore:
(df_csv
.write
.mode("overwrite")
.format("delta")
.saveAsTable("workspace.default.ticket_details_delta_table") #OBS! path = catalog.schema.table_name
)

# Read the Delta Table using python
spark.read.table("workspace.default.ticket_details_delta_table").display()

# Show all the tables under the schema workspace.default
spark.catalog.listTables("workspace.default")


### Bonus: Read the table using SQL statements in the cell

In [0]:
%sql
-- Read the Delta Table using SQL statement
select * from workspace.default.ticket_details_delta_table order by customer_id;

In [0]:
%sql
describe detail ticket_details_delta_table;

In [0]:
%sql
describe extended ticket_details_delta_table;

In [0]:
%sql
describe history ticket_details_delta_table;

### Create a DataFrame from a file

In [0]:
display(dbutils.fs.ls('/databricks-datasets/samples/'))
# Equivalent to %fs ls '/databricks-datasets' (run it in a solo cell)

In [0]:
df_population = (spark.read
  .format("csv")
  .option("header", True)
  .option("inferSchema", True)
  .load("/databricks-datasets/samples/population-vs-price/data_geo.csv")
)
display(df_population)

### Create a DataFrame from a JSON response
Skipping this one for now

##Data Manipulation Language (DML): Transform data with DataFrames

### Column operations

#### Select columns

In [0]:
"""
6 different ways to select columns
"""
# Output all the columns in a DataFrame
df_population.columns

# Select a single column using a string literal. Straight-forward.
df_population.select("city") # output: DataFrame[city: string]
#OBS! Select is not the same as display/print
display(df_population.select("city")) # output: content of the column

# Select multiple columns
df_population.select("city", "price", "population")

# Select using the pyspark.sql.functions.col()
from pyspark.sql.functions import col
df_population.select(col("city"))

# Select using the pyspark.sql.functions.expr()
from pyspark.sql.functions import expr
df_population.select(expr("city"))

# Select using selectExpr() - accepts SQL expressions. E.g. Useful for aliases, aggregations
df_population.selectExpr("city as city_name")

# Select using [] operator - Series
df_population.select(df_population["city"])
df_population.select(
    df_population["city"],
    df_population["state"],
    df_population["state code"]
)
# Select using . operator - OBS! It does not work!!!
#df_population.select(df_population.city)

#### Create columns

In [0]:
volume_file_path = "/Volumes/workspace/default/learning/ticket_details.csv"
df_csv = (spark.read
  .format("csv")
  .option("header", True)
  .option("inferSchema", True)
  .load(volume_file_path)
)

df_flag = df_csv.withColumn("flag", col("price_per_ticket") > 300) # column 'flag' is added and contains boolean values based on the condition
display(df_flag)

#### Rename columns

In [0]:
df_flag_renamed = df_flag.withColumnRenamed("flag", "price_flag")

#### Cast column types

In [0]:
display(df_population.select("2015 median sales price"))

df_casted = df_population.withColumn("2015 median sales price", col("2015 median sales price").cast("decimal(10,2)"))
#OBS! Ojo al nuevo uso del mét withColumn
df_casted.describe

#### Remove columns

In [0]:
df_flag_renamed.drop("price_flag")

### Row operations

#### Filter rows

#### Remove duplicate rows

####Handle null values

####Append row - Union

####Sort rows - OrderBy

### Join DataFrames

### Aggregate data - GroupBy

### Export data

#251127: Cont´d practicing PySpark

##Modularization principles: Use of functions

In [0]:
# Function to load data into a data frame
def load_data (file_path):
    return spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

# Function to add a new column to an existing data frame
def add_column (df, column_name, column_value):
    return df.withColumn(column_name, lit(column_value)) #lit() is a function to create a literal value for all rows

# Function to add a new column
def add_column_v2 (df, column_name, column_value):
    return df.withColumn(column_name,
                         when(col(column_value)==0, 'Normal').otherwise('Unknown'))

In [0]:
#Main
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, when

# Sample data
data = [Row(id=1, status=0), Row(id=2, status=1), Row(id=3, status=0)]
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("status", IntegerType(), True)
])
df = spark.createDataFrame(data, schema)

# Add new column based on 'status'
df_new = add_column_v2(df, "status_label", "status")
display(df_new)

#251128: Unit Test for PySpark

In [0]:
#Given the function add_column_v2 above described,
#Create a unit test for it
import pyspark.testing.utils

def test_add_new_col():
    data = [(0,), (1,), (-1,), (None,)]
    columns = ["value"]
    df = spark.createDataFrame(data, columns)

    actual_df = add_column_v2(df, "new_col", "value")

    expected_data = [(0, "Normal"), (1, "Unkown"), (-1, "Unkown"), (None, "Unkown")]

    expected_df = spark.createDataFrame(expected_data, ["value", "new_col"])

    assertDataFrameEqual(actual_df, expected_df)